# Sentinel-2 NDVI around Halle: 2020 and 2021

This notebook performs the complete research workflow in Python: download raw Sentinel-2 L2A bands, calculate NDVI, apply the observation mask, crop with GeoPandas, visualise both years and export an optional local map layer. No frontend or JSON knowledge is required.

The selected observations are **24 June 2020** and **14 June 2021**. Both use the same 10 m EPSG:32631 grid. NDVI is calculated as `(B08 - B04) / (B08 + B04)`. Pixels with no data, a zero denominator, invalid NDVI, or SCL classes 0, 1, 3, 7, 8, 9, 10 and 11 are masked.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from greenwave_ndvi import (
    compute_ndvi, crop_to_bounds, download_raw_observation,
    export_categorical_layer, export_continuous_layer,
    municipality_bounds, open_raw_observation, prompt_cdse_credentials,
)
from greenwave_ndvi.source import DEFAULT_RAW_CACHE, raw_path

YEARS = (2020, 2021)
RAW_CACHE = Path(os.environ.get('GREENWAVE_NDVI_RAW_CACHE', DEFAULT_RAW_CACHE))
EXPORT_DIR = Path(os.environ.get('GREENWAVE_PLAYGROUND_EXPORT', '../../.cache/playground/web')).resolve()
OFFLINE = os.environ.get('GREENWAVE_NDVI_OFFLINE') == '1'

## 1. Download or reuse the raw bands

If a raw cache is missing, this cell reads `CDSE_SH_CLIENT_ID` and `CDSE_SH_CLIENT_SECRET` or asks for them interactively. The secret is hidden and never stored.

In [ ]:
missing = [year for year in YEARS if not raw_path(year, RAW_CACHE).exists()]
if missing and OFFLINE:
    raise FileNotFoundError(f'Offline mode is active and raw observations are missing: {missing}')
credentials = prompt_cdse_credentials() if missing else None
paths = {year: download_raw_observation(year, credentials, cache_dir=RAW_CACHE) for year in YEARS}
paths

## 2. Calculate NDVI in Python

The downloaded GeoTIFF contains raw red reflectance (`b04`), near-infrared reflectance (`b08`), the scene-classification layer (`scl`) and `data_mask`. `compute_ndvi` applies the formula and the documented validity mask locally.

In [ ]:
raw = {year: open_raw_observation(paths[year]) for year in YEARS}
ndvi = {year: compute_ndvi(raw[year]) for year in YEARS}
quality = pd.DataFrame([
    {
        'year': year,
        'date': raw[year].attrs['date'],
        'grid': f"{raw[year].sizes['x']} × {raw[year].sizes['y']}",
        'valid Zennevallei pixels (%)': round(float(np.isfinite(ndvi[year]).mean()) * 100, 3),
    }
    for year in YEARS
]).set_index('year')
quality

## 3. Crop a rectangle around Halle and compare the years

GeoPandas derives the rectangle from all Statbel sectors belonging to Halle and adds a 1 km margin. Both maps use the same extent and colour normalisation. Higher NDVI generally represents a stronger green vegetation signal, but does not by itself prove ecological quality or vegetation type.

In [ ]:
halle_bounds = municipality_bounds('Halle', padding_m=1000)
halle = {year: crop_to_bounds(ndvi[year], halle_bounds) for year in YEARS}

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True, constrained_layout=True)
for axis, year in zip(axes, YEARS):
    image = halle[year].plot(ax=axis, cmap='RdYlGn', vmin=-0.2, vmax=0.9, add_colorbar=False)
    axis.set_title(f"Halle NDVI · {raw[year].attrs['date']}")
    axis.set_aspect('equal')
fig.colorbar(image, ax=axes, shrink=0.78, label='NDVI')
plt.show()

In [ ]:
fig, axis = plt.subplots(figsize=(9, 4))
for year in YEARS:
    values = halle[year].values[np.isfinite(halle[year].values)]
    axis.hist(values, bins=80, density=True, histtype='step', linewidth=1.8, label=str(year))
axis.set(xlabel='NDVI', ylabel='Density', title='Valid NDVI distribution in the Halle rectangle')
axis.legend()
axis.grid(alpha=0.2)
plt.show()

## 4. Export the latest result to the local Test layer

This writes an ignored PNG and manifest. Start the map with `pnpm dev:playground-map` to display it. Nothing below is included in GitHub Pages.

In [ ]:
manifest_path = export_continuous_layer(
    halle[2021],
    title={'en': 'Halle NDVI 2021 test', 'nl': 'Halle NDVI 2021-test'},
    description={
        'en': 'NDVI calculated in Python from the 14 June 2021 Sentinel-2 observation.',
        'nl': 'NDVI berekend in Python op basis van de Sentinel-2-opname van 14 juni 2021.',
    },
    units='NDVI', vmin=-0.2, vmax=0.9, output_dir=EXPORT_DIR,
)
print(f'Local Test layer written to: {manifest_path}')

### Optional categorical export

The same bridge accepts any numeric classification. The example below is intentionally not executed and is not the dashboard's calibrated likely-vegetation method. Adapt the classes and threshold to the experiment, then run it to replace the local Test layer.

In [ ]:
# experimental = xr.where(halle[2021] >= 0.5, 1, 0).where(np.isfinite(halle[2021]))
# experimental.attrs.update(halle[2021].attrs)
# export_categorical_layer(
#     experimental,
#     classes=[
#         {'value': 0, 'label': 'Below experimental threshold', 'color': '#d9deda'},
#         {'value': 1, 'label': 'Above experimental threshold', 'color': '#238b45'},
#     ],
#     title='Experimental NDVI classification', output_dir=EXPORT_DIR,
# )